# COMMOT — Spatial cell-cell communication

Infers directional cell-cell communication from spatial transcriptomics data
using [COMMOT](https://github.com/zcang/COMMOT) and the
[CellChat](https://github.com/jinworks/CellChat) ligand-receptor database (mouse).

**Scope:** cells in meta-domain 6 (fibrotic scar), injury model `inj_1` only.
Slides with quality issues are excluded via the `exclude_batches` list.

**Distance threshold:** 50 µm — only cells within this radius are considered
as potential communication partners.

## Workflow

1. Subset to `inj_1` + meta-domain 6, exclude low-quality slides
2. Run `ct.tl.spatial_communication()` per slide → save one `.h5ad` per slide
3. Load all results and aggregate sender/receiver scores per cell type and timepoint
4. Analyse dominant pathways, temporal dynamics, and celltype_1–immune interactions
5. Visualise spatial communication maps per LR pair or pathway

**Input:** annotated spatial `.h5ad` with `obs['inj_type']`, `obs['meta_domain']`,
`obs['batch']`, `obs['celltype']`, and `obsm['spatial']`

**Output:** per-slide `*_commot.h5ad` + pathway/LR pair summary plots (PDF)

In [ ]:
import commot as ct
import scanpy as sc
import pandas as pd
import pandas as np

import matplotlib.pyplot as plt

## 1 · Run COMMOT per slide 

In [ ]:
adata = sc.read_h5ad("/path/to/your/annotated_spatial.h5ad")

In [ ]:
adata = sc.read_h5ad("/path/to/your/commot_results/")

In [ ]:
# Load the database
df_ligrec = ct.pp.ligand_receptor_database(database='CellChat', species='mouse')

# Check it loaded correctly
print(df_ligrec.shape)
print(df_ligrec.head())

In [ ]:
results = {}

import os
output_dir = '/path/to/your/commot_results/'
os.makedirs(output_dir, exist_ok=True)

for slide in adata_crush_md6.obs['batch'].unique():
    
    adata_slide = adata_crush_md6[adata_crush_md6.obs['batch'] == slide].copy()
    print(f"\nRunning COMMOT on slide {slide}: {adata_slide.shape[0]} cells")
    
    ct.tl.spatial_communication(
    adata_slide,
    database_name='CellChat',
    df_ligrec=df_ligrec,
    dis_thr=50,
    heteromeric=True,
    heteromeric_rule='min',
    pathway_sum=True,
    cost_type='euc'
)
    
    # Save immediately after each slide
    adata_slide.write_h5ad(f'{output_dir}/{slide}_commot.h5ad')
    print(f"Saved: {slide}")

## 2 · Load results

In [ ]:
import os
import glob
output_dir = '/path/to/your/commot_results/'

results = {}

for f in glob.glob(f'{output_dir}/*.h5ad'):
    slide = os.path.basename(f).replace('_commot.h5ad', '')
    results[slide] = sc.read_h5ad(f)
    print(f"Loaded {slide}: {results[slide].shape[0]} cells")

print(f"\nTotal slides loaded: {len(results)}")

In [ ]:
# Pick first slide
example = list(results.values())[0]

print("obsm keys:", list(example.obsm.keys()))
print("uns keys:", list(example.uns.keys()))

In [ ]:
import pandas as pd

# Check what pathways/LR pairs are available
example = list(results.values())[0]
print(example.uns['commot-CellChat-info'])

# Check sender/receiver score columns (one per LR pair + pathway summaries)
sender_df = pd.DataFrame(
    example.obsm['commot-CellChat-sum-sender'],
    index=example.obs_names
)


In [ ]:
sender_all = []
receiver_all = []

for slide, adata_slide in results.items():
    
    # Sender scores
    df_s = pd.DataFrame(
        adata_slide.obsm['commot-CellChat-sum-sender'],
        index=adata_slide.obs_names
    )
    df_s['batch'] = slide
    df_s['celltype'] = adata_slide.obs['celltype'].values  # adjust column name
    sender_all.append(df_s)
    
    # Receiver scores
    df_r = pd.DataFrame(
        adata_slide.obsm['commot-CellChat-sum-receiver'],
        index=adata_slide.obs_names
    )
    df_r['batch'] = slide
    df_r['celltype'] = adata_slide.obs['celltype'].values  # adjust column name
    receiver_all.append(df_r)

sender_all = pd.concat(sender_all)
receiver_all = pd.concat(receiver_all)



## 3 · Aggregate sender / receiver scores

In [ ]:
# Get LR columns separately for each dataframe
sender_lr_cols = [col for col in sender_all.columns if col not in ['batch', 'celltype']]
receiver_lr_cols = [col for col in receiver_all.columns if col not in ['batch', 'celltype']]

# Average sender scores per cell type per slide
sender_summary = sender_all.groupby(['batch', 'celltype'])[sender_lr_cols].mean().reset_index()

# Average receiver scores per cell type per slide
receiver_summary = receiver_all.groupby(['batch', 'celltype'])[receiver_lr_cols].mean().reset_index()

print(sender_summary.shape)
print(receiver_summary.shape)

In [ ]:
# add day as metadata

timepoint_order = ['d2', 'd3', 'd4', 'd5', 'd7', 'd9', 'd14']

for df in [sender_summary, receiver_summary]:
    df['day'] = df['batch'].str.extract(r'(d\d+)')
    df['day'] = pd.Categorical(df['day'], categories=timepoint_order, ordered=True)

print(sender_summary['day'].value_counts().sort_index())
print(sender_summary['celltype'].unique())

In [ ]:
sender_summary['celltype'].unique()

## 4 · Analysis

### 4.1 · Main pathways in meta-domain 6

In [ ]:
sender_summary = pd.read_csv(f'{output_dir}/sender_summary.csv')
receiver_summary = pd.read_csv(f'{output_dir}/receiver_summary.csv')

In [ ]:
# Question 1 — Main pathways in metadomain 6
overall_sender = sender_summary[sender_pathway_cols].mean().sort_values(ascending=False)
overall_receiver = receiver_summary[receiver_pathway_cols].mean().sort_values(ascending=False)

print("Top sender pathways:\n", overall_sender)
print("\nTop receiver pathways:\n", overall_receiver)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

overall_sender.plot(kind='bar', ax=axes[0], color='indianred')
axes[0].set_title('Top sender pathways - metadomain 6')
axes[0].set_ylabel('Mean communication score')
axes[0].set_xticklabels([c.replace('s-', '') for c in overall_sender.index], rotation=45, ha='right')

overall_receiver.plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Top receiver pathways - metadomain 6')
axes[1].set_ylabel('Mean communication score')
axes[1].set_xticklabels([c.replace('r-', '') for c in overall_receiver.index], rotation=45, ha='right')

fig.tight_layout()
fig.savefig(f'{output_dir}/top_pathways_metadomain6.pdf', bbox_inches='tight', facecolor='white')
plt.show()



In [ ]:
# Average per timepoint across all cell types
sender_tp = sender_summary.groupby('day')[sender_pathway_cols].mean()
receiver_tp = receiver_summary.groupby('day')[receiver_pathway_cols].mean()

# Plot heatmaps: rows = pathways, columns = timepoints
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

im1 = axes[0].imshow(sender_tp.T, aspect='auto', cmap='RdBu_r')
axes[0].set_xticks(range(len(sender_tp.index)))
axes[0].set_xticklabels(sender_tp.index, rotation=45, ha='right')
axes[0].set_yticks(range(len(sender_pathway_cols)))
axes[0].set_yticklabels([c.replace('s-', '') for c in sender_pathway_cols])
axes[0].set_title('Sender pathways per timepoint - metadomain 6')
plt.colorbar(im1, ax=axes[0], label='Mean score')

im2 = axes[1].imshow(receiver_tp.T, aspect='auto', cmap='RdBu_r')
axes[1].set_xticks(range(len(receiver_tp.index)))
axes[1].set_xticklabels(receiver_tp.index, rotation=45, ha='right')
axes[1].set_yticks(range(len(receiver_pathway_cols)))
axes[1].set_yticklabels([c.replace('r-', '') for c in receiver_pathway_cols])
axes[1].set_title('Receiver pathways per timepoint - metadomain 6')
plt.colorbar(im2, ax=axes[1], label='Mean score')

fig.tight_layout()
fig.savefig(f'{output_dir}/pathways_per_timepoint_metadomain6.pdf', bbox_inches='tight', facecolor='white')
plt.show()

### 4.2 · celltype_1 vs immune interactions per timepoint

In [ ]:
ct1_sender_tp = sender_summary[sender_summary['celltype'] == 'celltype_1'].groupby('day')[sender_pathway_cols].mean()
immune_sender_tp = sender_summary[sender_summary['celltype'] == 'celltype_2'].groupby('day')[sender_pathway_cols].mean()
ct1_receiver_tp = receiver_summary[receiver_summary['celltype'] == 'celltype_1'].groupby('day')[receiver_pathway_cols].mean()
immune_receiver_tp = receiver_summary[receiver_summary['celltype'] == 'celltype_2'].groupby('day')[receiver_pathway_cols].mean()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

titles = [
    ('celltype_1 — Sender', ct1_sender_tp, sender_pathway_cols, 's-'),
    ('celltype_2 — Sender', immune_sender_tp, sender_pathway_cols, 's-'),
    ('celltype_1 — Receiver', ct1_receiver_tp, receiver_pathway_cols, 'r-'),
    ('celltype_2 — Receiver', immune_receiver_tp, receiver_pathway_cols, 'r-'),
]

for ax, (title, df, cols, prefix) in zip(axes.flatten(), titles):
    im = ax.imshow(df.T, aspect='auto', cmap='Reds')
    ax.set_xticks(range(len(df.index)))
    ax.set_xticklabels(df.index)
    ax.set_yticks(range(len(cols)))
    ax.set_yticklabels([c.replace(prefix, '') for c in cols], fontsize=9)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label='Score')

fig.tight_layout()
fig.savefig(f'{output_dir}/ct1_ct2_communication.pdf', bbox_inches='tight', facecolor='white')
plt.show()

### 4.3 · Total sent / received signals per timepoint and cell type 

In [ ]:
# d1 excluded as too low values
# normalization is per slide number

In [ ]:
# Define phase mapping
phase_map = {
    'd3': 'acute', 'd2': 'acute',
    'd4': 'subacute', 'd5': 'subacute', 'd7': 'subacute',
    'd9': 'late', 'd14': 'late'
}
phase_order = ['acute', 'subacute', 'late']

sender_summary['phase'] = sender_summary['day'].map(phase_map)
receiver_summary['phase'] = receiver_summary['day'].map(phase_map)

sender_summary['phase'] = pd.Categorical(sender_summary['phase'], categories=phase_order, ordered=True)
receiver_summary['phase'] = pd.Categorical(receiver_summary['phase'], categories=phase_order, ordered=True)

In [ ]:
def plot_total_signal(sender_summary, receiver_summary,
                      total_sender_col, total_receiver_col,
                      output_dir,
                      mode='raw',          # 'raw', 'normalized', 'zscore'
                      groupby='day',       # 'day' or 'phase'
                      plot_type='heatmap'  # 'heatmap' or 'lineplot'
                      ):
    from scipy.stats import zscore as scipy_zscore

    # --- Phase mapping ---
    if groupby == 'phase':
        phase_map = {
            'd2': 'acute', 'd3': 'acute',
            'd4': 'subacute', 'd5': 'subacute', 'd7': 'subacute',
            'd9': 'late', 'd14': 'late'
        }
        phase_order = ['acute', 'subacute', 'late']
        for df in [sender_summary, receiver_summary]:
            df['phase'] = pd.Categorical(df['day'].map(phase_map), 
                                         categories=phase_order, ordered=True)
        group_col = 'phase'
    else:
        group_col = 'day'

    # --- Aggregate ---
    sender_total = sender_summary.groupby([group_col, 'celltype'])[total_sender_col].mean().reset_index()
    receiver_total = receiver_summary.groupby([group_col, 'celltype'])[total_receiver_col].mean().reset_index()

    sender_pivot = sender_total.pivot(index=group_col, columns='celltype', values=total_sender_col).dropna()
    receiver_pivot = receiver_total.pivot(index=group_col, columns='celltype', values=total_receiver_col).dropna()

    # --- Normalize ---
    if mode == 'raw':
        sender_plot = sender_pivot
        receiver_plot = receiver_pivot
        label = 'Mean score (raw)'
        cmap_s, cmap_r = 'Reds', 'Reds'
        vmin, vmax = None, None

    elif mode == 'normalized':
        n_slides = sender_summary.groupby(group_col)['batch'].nunique()
        sender_plot = sender_pivot.div(n_slides, axis=0)
        receiver_plot = receiver_pivot.div(n_slides, axis=0)
        label = 'Mean score / n slides'
        cmap_s, cmap_r = 'Reds', 'Reds'
        vmin, vmax = None, None

    elif mode == 'zscore':
        sender_plot = sender_pivot.apply(scipy_zscore, axis=0)
        receiver_plot = receiver_pivot.apply(scipy_zscore, axis=0)
        label = 'Z-score'
        cmap_s = cmap_r = 'RdBu_r'
        vmax = max(sender_plot.abs().max().max(), receiver_plot.abs().max().max())
        vmin = -vmax

    else:
        raise ValueError("mode must be 'raw', 'normalized', or 'zscore'")

    # --- Plot ---
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    title_suffix = f'({mode} | {groupby})'

    if plot_type == 'heatmap':
        im1 = axes[0].imshow(sender_plot.T, aspect='auto', cmap=cmap_s, vmin=vmin, vmax=vmax)
        axes[0].set_xticks(range(len(sender_plot.index)))
        axes[0].set_xticklabels(sender_plot.index, rotation=45, ha='right')
        axes[0].set_yticks(range(len(sender_plot.columns)))
        axes[0].set_yticklabels(sender_plot.columns)
        axes[0].set_title(f'Total signal SENT {title_suffix}')
        plt.colorbar(im1, ax=axes[0], label=label)

        im2 = axes[1].imshow(receiver_plot.T, aspect='auto', cmap=cmap_r, vmin=vmin, vmax=vmax)
        axes[1].set_xticks(range(len(receiver_plot.index)))
        axes[1].set_xticklabels(receiver_plot.index, rotation=45, ha='right')
        axes[1].set_yticks(range(len(receiver_plot.columns)))
        axes[1].set_yticklabels(receiver_plot.columns)
        axes[1].set_title(f'Total signal RECEIVED {title_suffix}')
        plt.colorbar(im2, ax=axes[1], label=label)

    elif plot_type == 'lineplot':
        for cell_type in sender_plot.columns:
            axes[0].plot(range(len(sender_plot.index)), sender_plot[cell_type], marker='o', label=cell_type)
        axes[0].set_xticks(range(len(sender_plot.index)))
        axes[0].set_xticklabels(sender_plot.index, rotation=45, ha='right')
        axes[0].set_ylabel(label)
        axes[0].set_title(f'Total signal SENT {title_suffix}')
        axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)

        for cell_type in receiver_plot.columns:
            axes[1].plot(range(len(receiver_plot.index)), receiver_plot[cell_type], marker='o', label=cell_type)
        axes[1].set_xticks(range(len(receiver_plot.index)))
        axes[1].set_xticklabels(receiver_plot.index, rotation=45, ha='right')
        axes[1].set_ylabel(label)
        axes[1].set_title(f'Total signal RECEIVED {title_suffix}')
        axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)

    else:
        raise ValueError("plot_type must be 'heatmap' or 'lineplot'")

    fig.suptitle(f'Metadomain 6 — inj_1 injury', fontsize=13, y=1.02)
    fig.tight_layout()

    filename = f'{output_dir}/total_signal_{groupby}_{mode}_{plot_type}.pdf'
    fig.savefig(filename, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"Saved: {filename}")

In [ ]:
plot_total_signal(sender_summary, receiver_summary,
                  's-total-total', 'r-total-total',
                  output_dir, mode='normalized', groupby='phase', plot_type='heatmap')

## 5 · Spatial visualization per slide

In [ ]:
for pathway, group in df_ligrec.groupby('pathway'):
    print(f"\n{pathway}:")
    for _, row in group.iterrows():
        print(f"  {row['ligand']} → {row['receptor']}")

In [ ]:
import io
from PIL import Image
from PIL import Image, ImageDraw, ImageFont


slide_name = 'your_slide_id'  # replace with a slide ID from your dataset
adata_slide = sc.read_h5ad(f'{output_dir}/{slide_name}_commot.h5ad')
output_dir = '/path/to/your/commot_results/'
lr = ('Tgfb1', 'Tgfbr1_Tgfbr2')


ct.tl.communication_direction(
    adata_slide,
    database_name='CellChat',
    lr_pair=lr,
    k=5
)

In [ ]:
def capture_commot_plot(adata_slide, lr, summary, cmap):
    ct.pl.plot_cell_communication(
        adata_slide, database_name='CellChat', lr_pair=lr,
        plot_method='grid', 
        background_legend=True, 
        scale=0.00003, 
        ndsize=20,
        grid_density=0.5, 
        summary=summary, 
        background='summary',
        clustering='cell_type', 
        cmap=cmap, 
        normalize_v=True, 
        normalize_v_quantile=0.995
    )
    buf = io.BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight', facecolor='white', dpi=150)
    plt.close()
    buf.seek(0)
    return Image.open(buf).copy()

def add_title_to_image(img, title, font_size=20):
    """Add a white banner with title on top of image."""
    banner_height = font_size + 10
    new_img = Image.new('RGB', (img.width, img.height + banner_height), (255, 255, 255))
    new_img.paste(img, (0, banner_height))
    draw = ImageDraw.Draw(new_img)
    # Try to use a nice font, fallback to default
    try:
        font = ImageFont.truetype('/System/Library/Fonts/Helvetica.ttc', font_size)
    except:
        font = ImageFont.load_default()
    text_width = draw.textlength(title, font=font)
    x = (img.width - text_width) / 2
    draw.text((x, 10), title, fill=(0, 0, 0), font=font)
    return new_img

# Capture plots
img_sender = capture_commot_plot(adata_slide, lr, 'sender', 'Blues')
img_receiver = capture_commot_plot(adata_slide, lr, 'receiver', 'Reds')

# Add titles
lr_str = f'{lr[0]} → {lr[1]}'
img_sender = add_title_to_image(img_sender, f'Sender | {lr_str} | {slide_name}')
img_receiver = add_title_to_image(img_receiver, f'Receiver | {lr_str} | {slide_name}')

# Combine side by side
total_width = img_sender.width + img_receiver.width
max_height = max(img_sender.height, img_receiver.height)
combined = Image.new('RGB', (total_width, max_height), (255, 255, 255))
combined.paste(img_sender, (0, 0))
combined.paste(img_receiver, (img_sender.width, 0))

# Save as PDF
fig, ax = plt.subplots(figsize=(22, 9))
ax.imshow(combined)
ax.axis('off')
fig.tight_layout()
fig.savefig(f'{output_dir}/{slide_name}_{lr[0]}_{lr[1]}_sender_receiver.pdf',
            bbox_inches='tight', facecolor='white')
plt.show()
print(f"Saved: {slide_name}_{lr[0]}_{lr[1]}_sender_receiver.pdf")

### 5.1 · All LR pairs within a pathway

In [ ]:
def plot_communication_pathway(adata_slide, df_ligrec, pathway_name, slide_name, output_dir,
                                scale=0.00003, ndsize=8, grid_density=0.5):
    from PIL import Image, ImageDraw, ImageFont
    import io

    # Get all LR pairs for the pathway
    lr_pairs = df_ligrec[df_ligrec['pathway'] == pathway_name][['ligand', 'receptor']].values.tolist()
    print(f"Found {len(lr_pairs)} LR pairs for pathway {pathway_name}: {lr_pairs}")

    def capture_commot_plot(adata_slide, lr, summary, cmap):
        ct.tl.communication_direction(adata_slide, database_name='CellChat', lr_pair=tuple(lr), k=5)
        ct.pl.plot_cell_communication(
            adata_slide, database_name='CellChat', lr_pair=tuple(lr),
            plot_method='grid', background_legend=True, scale=scale, ndsize=ndsize,
            grid_density=grid_density, summary=summary, background='summary',
            clustering='celltype', cmap=cmap, normalize_v=True, normalize_v_quantile=0.995
        )
        buf = io.BytesIO()
        plt.savefig(buf, format='png', bbox_inches='tight', facecolor='white', dpi=150)
        plt.close()
        buf.seek(0)
        return Image.open(buf).copy()

    def add_title_to_image(img, title, font_size=20):
        banner_height = font_size + 20
        new_img = Image.new('RGB', (img.width, img.height + banner_height), (255, 255, 255))
        new_img.paste(img, (0, banner_height))
        draw = ImageDraw.Draw(new_img)
        try:
            font = ImageFont.truetype('/System/Library/Fonts/Arial.ttc', font_size)
        except:
            font = ImageFont.load_default()
        text_width = draw.textlength(title, font=font)
        x = (img.width - text_width) / 2
        draw.text((x, 10), title, fill=(0, 0, 0), font=font)
        return new_img

    # Build one row per LR pair (sender | receiver)
    row_images = []
    for lr in lr_pairs:
        lr_str = f'{lr[0]} → {lr[1]}'
        print(f"  Plotting {lr_str}...")

        img_sender = capture_commot_plot(adata_slide, lr, 'sender', 'Blues')
        img_receiver = capture_commot_plot(adata_slide, lr, 'receiver', 'Reds')

        img_sender = add_title_to_image(img_sender, f'Sender | {lr_str} | {slide_name}')
        img_receiver = add_title_to_image(img_receiver, f'Receiver | {lr_str} | {slide_name}')

        # Combine sender and receiver side by side
        row_width = img_sender.width + img_receiver.width
        row_height = max(img_sender.height, img_receiver.height)
        row = Image.new('RGB', (row_width, row_height), (255, 255, 255))
        row.paste(img_sender, (0, 0))
        row.paste(img_receiver, (img_sender.width, 0))
        row_images.append(row)

    # Stack all rows vertically
    total_width = max(r.width for r in row_images)
    total_height = sum(r.height for r in row_images)
    combined = Image.new('RGB', (total_width, total_height), (255, 255, 255))
    y_offset = 0
    for row in row_images:
        combined.paste(row, (0, y_offset))
        y_offset += row.height

    # Save
    fig, ax = plt.subplots(figsize=(22, 9 * len(lr_pairs)))
    ax.imshow(combined)
    ax.axis('off')
    fig.tight_layout()
    fname = f'{output_dir}/{slide_name}_{pathway_name}_all_pairs.pdf'
    fig.savefig(fname, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"Saved: {fname}")

In [ ]:
plot_communication_pathway(
    adata_slide=adata_slide,
    df_ligrec=df_ligrec,
    pathway_name='ANGPT',
    slide_name=slide_name,
    output_dir=output_dir
)

### 5.2 · Aggregated pathway summary 

In [ ]:
def plot_communication_pathway_summary(adata_slide, df_ligrec, pathway_name, slide_name, output_dir,
                                        mode='average',
                                        scale=0.00003,
                                        ndsize=20,
                                        grid_density=0.5,
                                        grid_width=0.002,
                                        arrow_color='#333333'):
    import numpy as np
    import io
    from PIL import Image, ImageDraw, ImageFont

    # --- Get all LR pairs for the pathway ---
    lr_pairs = df_ligrec[df_ligrec['pathway'] == pathway_name][['ligand', 'receptor']].values.tolist()
    print(f"Found {len(lr_pairs)} LR pairs for pathway '{pathway_name}'")

    if len(lr_pairs) == 0:
        raise ValueError(f"No LR pairs found for pathway '{pathway_name}'")

    # --- Filter to only pairs actually computed in this slide ---
    available_obsp = list(adata_slide.obsp.keys())
    lr_pairs = [lr for lr in lr_pairs if f'commot-CellChat-{lr[0]}-{lr[1]}' in available_obsp]
    print(f"After filtering for available pairs in slide: {len(lr_pairs)} pairs")
    for lr in lr_pairs:
        print(f"  {lr[0]} → {lr[1]}")

    if len(lr_pairs) == 0:
        raise ValueError(f"No computed LR pairs found for pathway '{pathway_name}' in slide '{slide_name}'")

    # --- Compute vector fields for available LR pairs ---
    for lr in lr_pairs:
        print(f"  Computing direction for {lr[0]} → {lr[1]}...")
        ct.tl.communication_direction(adata_slide, database_name='CellChat', lr_pair=tuple(lr), k=5)

    # --- Collect and aggregate scores ---
    sender_df = pd.DataFrame(adata_slide.obsm['commot-CellChat-sum-sender'], index=adata_slide.obs_names)
    receiver_df = pd.DataFrame(adata_slide.obsm['commot-CellChat-sum-receiver'], index=adata_slide.obs_names)

    sender_scores, receiver_scores = [], []
    for lr in lr_pairs:
        s_col = [c for c in sender_df.columns if lr[0] in c and lr[1] in c]
        r_col = [c for c in receiver_df.columns if lr[0] in c and lr[1] in c]
        if s_col:
            sender_scores.append(sender_df[s_col[0]].values)
        if r_col:
            receiver_scores.append(receiver_df[r_col[0]].values)

    sender_scores = np.array(sender_scores)
    receiver_scores = np.array(receiver_scores)

    if mode == 'average':
        sender_agg = sender_scores.mean(axis=0)
        receiver_agg = receiver_scores.mean(axis=0)
    elif mode == 'max':
        sender_agg = sender_scores.max(axis=0)
        receiver_agg = receiver_scores.max(axis=0)
    else:
        raise ValueError("mode must be 'average' or 'max'")

    # --- Store aggregated scores ---
    adata_slide.obsm['commot-CellChat-sum-sender'][f's-{pathway_name}-agg'] = sender_agg
    adata_slide.obsm['commot-CellChat-sum-receiver'][f'r-{pathway_name}-agg'] = receiver_agg

    proxy_lr = tuple(lr_pairs[0])

    # --- Helper: capture plot as image ---
    def capture_plot(summary, cmap):
        ct.pl.plot_cell_communication(
            adata_slide,
            database_name='CellChat',
            lr_pair=proxy_lr,
            plot_method='grid',
            background_legend=True,
            scale=scale,
            ndsize=ndsize,
            grid_density=grid_density,
            grid_width=grid_width,
            arrow_color=arrow_color,
            summary=summary,
            background='summary',
            clustering='cell_type',
            cmap=cmap,
            normalize_v=True,
            normalize_v_quantile=0.995
        )
        buf = io.BytesIO()
        plt.savefig(buf, format='png', bbox_inches='tight', facecolor='white', dpi=150)
        plt.close()
        buf.seek(0)
        return Image.open(buf).copy()

    # --- Helper: add title banner ---
    def add_title(img, title, font_size=30):
        banner_height = font_size + 20
        new_img = Image.new('RGB', (img.width, img.height + banner_height), (255, 255, 255))
        new_img.paste(img, (0, banner_height))
        draw = ImageDraw.Draw(new_img)
        try:
            font = ImageFont.truetype('/System/Library/Fonts/Helvetica.ttc', font_size)
        except:
            font = ImageFont.load_default()
        text_width = draw.textlength(title, font=font)
        draw.text(((img.width - text_width) / 2, 10), title, fill=(0, 0, 0), font=font)
        return new_img

    # --- Capture sender and receiver ---
    print("Plotting sender...")
    img_sender = capture_plot('sender', 'Blues')
    img_sender = add_title(img_sender, f'Sender | {pathway_name} ({mode}) | {slide_name}')

    print("Plotting receiver...")
    img_receiver = capture_plot('receiver', 'Reds')
    img_receiver = add_title(img_receiver, f'Receiver | {pathway_name} ({mode}) | {slide_name}')

    # --- Combine side by side ---
    total_width = img_sender.width + img_receiver.width
    max_height = max(img_sender.height, img_receiver.height)
    combined = Image.new('RGB', (total_width, max_height), (255, 255, 255))
    combined.paste(img_sender, (0, 0))
    combined.paste(img_receiver, (img_sender.width, 0))

    # --- Save ---
    fig, ax = plt.subplots(figsize=(22, 9))
    ax.imshow(combined)
    ax.axis('off')
    fig.tight_layout()
    fname = f'{output_dir}/{slide_name}_{pathway_name}_{mode}_summary.pdf'
    fig.savefig(fname, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"Saved: {fname}")

In [ ]:
plot_communication_pathway_summary(
    adata_slide=adata_slide,
    df_ligrec=df_ligrec,
    pathway_name='CSF',
    slide_name=slide_name,
    output_dir=output_dir,
    mode='average',
    scale=0.00003,
    ndsize=40,
    grid_density=0.5,
    grid_width=0.003,
    arrow_color='#333333'
)